## Exploration of the Green Taxi data
Goal: find out what must be cleaned before we can sum **revenue per day**.
Exploration only. The pipeline itself lives in `src/`.

Plan: 1) load one file, 2) pick the columns, 3) check types, 4) check missing values, 5) check value ranges, 6) write down cleaning rules.

In [1]:
import pandas as pd
pd.set_option("display.max_columns", 50)

df = pd.read_parquet("../data/raw/green_tripdata_2025-01.parquet")
print(df.shape)
df.head()

(48326, 21)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,N,1.0,75,235,1.0,5.93,24.70,1.0,0.5,6.80,0.00,NaN,1.0,34.00,1.0,1.0,0.00,0.0
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,N,1.0,166,75,1.0,1.32,8.60,1.0,0.5,0.00,0.00,NaN,1.0,11.10,2.0,1.0,0.00,0.0
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,N,5.0,171,73,1.0,0.41,25.55,0.0,0.0,0.00,0.00,NaN,1.0,26.55,2.0,2.0,0.00,0.0
3,2,2025-01-01 00:52:24,2025-01-01 01:07:52,N,1.0,74,223,1.0,4.12,21.20,1.0,0.5,6.13,6.94,NaN,1.0,36.77,1.0,1.0,0.00,0.0
4,2,2025-01-01 00:25:05,2025-01-01 01:01:10,N,1.0,66,158,1.0,4.71,33.80,1.0,0.5,7.81,0.00,NaN,1.0,46.86,1.0,1.0,2.75,0.0


## 1. Which columns do we need?
When did the trip happen, and what was paid? Look at the schema and the TLC data dictionary.

In [2]:
df.dtypes

VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag                  str
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
dtype: object

Candidates:
- **When:** `lpep_pickup_datetime` or `lpep_dropoff_datetime`. *Which one defines the "day" of a trip? Write down your choice and why.*
- **Paid:** `total_amount` or `fare_amount`? *What is the difference? Which one is "revenue"? (Compare with tip, tolls, taxes, surcharges.)*

In [3]:
cols = ["lpep_pickup_datetime", "lpep_dropoff_datetime", "fare_amount", "tip_amount", "total_amount"]
df[cols].describe()

,lpep_pickup_datetime,lpep_dropoff_datetime,fare_amount,tip_amount,total_amount
count,48326,48326,48326.000000,48326.000000,48326.000000
mean,2025-01-17 04:19:01.965753,2025-01-17 04:38:59.277448,16.762466,2.481859,22.634242
min,2024-12-25 23:13:15,2024-12-25 23:13:17,-113.000000,-0.900000,-114.000000
25%,2025-01-09 14:00:31.500000,2025-01-09 14:26:33.500000,9.300000,0.000000,13.705000
50%,2025-01-16 19:46:55.500000,2025-01-16 20:01:19.500000,13.500000,2.070000,18.750000
75%,2025-01-24 17:47:35.500000,2025-01-24 18:03:52,19.100000,3.690000,26.465000
max,2025-02-05 18:46:24,2025-02-05 19:11:47,336.200000,252.050000,371.400000
std,NaN,NaN,13.308342,3.213612,15.435061


## 2. Does total_amount add up?
Check whether `total_amount` equals the sum of its components. Which payment types are there?

In [4]:
parts = ["fare_amount","extra","mta_tax","tip_amount","tolls_amount","improvement_surcharge","congestion_surcharge","cbd_congestion_fee"]
diff = df["total_amount"] - df[parts].fillna(0).sum(axis=1)
print((diff.abs() > 0.01).sum(), "rows where total != sum of parts")
df["payment_type"].value_counts(dropna=False)

6698 rows where total != sum of parts


payment_type
1.0    34635
2.0    11424
NaN     1836
3.0      332
4.0       98
5.0        1
Name: count, dtype: int64

In [13]:
df["diff"] = (df["total_amount"] - df[parts].fillna(0).sum(axis=1)).round(2)
mismatch = df[df["diff"].abs() > 0.01]
print("mismatch rows:", len(mismatch), "| net effect on revenue:", round(mismatch["diff"].sum(), 2))
print(mismatch["diff"].value_counts().head(6))
mismatch.groupby("diff")[["mta_tax", "extra", "congestion_surcharge", "cbd_congestion_fee"]].agg(lambda c: c.value_counts(dropna=False).index[0]).loc[[-1.0, -3.75, -4.5, 2.75, 3.5]]

mismatch rows: 6698 | net effect on revenue: -8841.9
diff
-1.00    4467
-3.75    1220
-4.50     373
 3.50     292
 2.75     283
 2.50      19
Name: count, dtype: int64


,mta_tax,extra,congestion_surcharge,cbd_congestion_fee
diff,,,,
-1.00,1.5,0.00,0.00,0.00
-3.75,1.5,2.75,2.75,0.00
-4.50,1.5,3.50,2.75,0.75
2.75,0.5,0.00,NaN,NaN
3.50,0.5,0.00,NaN,NaN


**Finding:** the mismatches come in fixed sizes, and they trace back to a few oddities in the components, not to errors in `total_amount`:
- `-1.00`: `mta_tax` is recorded as 1.5, but the total only includes the usual 0.5.
- `-3.75` / `-4.50`: as above, and `extra` already contains the congestion surcharge (2.75) and the CBD fee (0.75), so adding them again double-counts.
- `+2.75` / `+3.50`: rows with no payment type, where `congestion_surcharge` and `cbd_congestion_fee` are NaN but the total includes them.

`total_amount` is the figure the customer was charged, so we use it as revenue. The component columns are not reliable enough to rebuild it. (The reasons above are inferred from the arithmetic, not from TLC documentation.)

## 3. Missing values

In [5]:
df.isna().sum()[lambda s: s > 0]

store_and_fwd_flag       1836
RatecodeID               1836
passenger_count          1836
ehail_fee               48326
payment_type             1836
trip_type                1843
congestion_surcharge     1836
cbd_congestion_fee       1836
dtype: int64

*Are the columns we need ever missing? Are the other missing columns relevant for revenue?*

## 4. Date ranges
The file is called **2025-01**. Does it only contain January 2025?

In [6]:
print(df["lpep_pickup_datetime"].min(), "->", df["lpep_pickup_datetime"].max())
df["lpep_pickup_datetime"].dt.to_period("M").value_counts().sort_index()

2024-12-25 23:13:15 -> 2025-02-05 18:46:24


lpep_pickup_datetime
2024-12        6
2025-01    48283
2025-02       37
Freq: M, Name: count, dtype: int64

*Which rows fall outside January? Show a few of them. What should happen with them?*

In [7]:
outside = df[df["lpep_pickup_datetime"].dt.to_period("M") != "2025-01"]
print(len(outside), "rows outside January 2025")
print("total_amount of those rows:", round(outside["total_amount"].sum(), 2))
outside[["lpep_pickup_datetime", "lpep_dropoff_datetime", "total_amount"]].sort_values("lpep_pickup_datetime").head(10)

43 rows outside January 2025
total_amount of those rows: 949.48


,lpep_pickup_datetime,lpep_dropoff_datetime,total_amount
320,2024-12-25 23:13:15,2024-12-25 23:13:17,43.20
4108,2024-12-29 01:23:41,2024-12-29 01:41:08,72.94
251,2024-12-31 19:56:14,2024-12-31 20:03:33,13.32
9,2024-12-31 22:42:13,2024-12-31 22:42:31,25.00
10,2024-12-31 23:01:11,2024-12-31 23:04:29,13.81
73,2024-12-31 23:30:28,2024-12-31 23:35:39,11.90
44849,2025-02-01 01:55:11,2025-02-01 02:05:28,18.36
44857,2025-02-01 02:09:14,2025-02-01 02:24:50,17.40
45217,2025-02-01 11:45:49,2025-02-01 11:56:11,15.70
45365,2025-02-01 13:06:08,2025-02-01 13:17:43,20.60


## 5. Amount ranges

In [8]:
df["total_amount"].describe()

count    48326.000000
mean        22.634242
std         15.435061
min       -114.000000
25%         13.705000
50%         18.750000
75%         26.465000
max        371.400000
Name: total_amount, dtype: float64

In [9]:
print("negative:", (df["total_amount"] < 0).sum())
print("zero:", (df["total_amount"] == 0).sum())
df[df["total_amount"] < 0].head()

negative: 153
zero: 33


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
245,2,2025-01-01 04:20:27,2025-01-01 04:40:48,N,5.0,82,82,1.0,0.00,-113.00,0.0,0.0,0.0,0.0,NaN,-1.0,-114.00,4.0,1.0,0.0,0.0
561,2,2025-01-01 15:51:18,2025-01-01 15:51:28,N,1.0,166,166,1.0,0.01,-3.00,0.0,-0.5,0.0,0.0,NaN,-1.0,-4.50,3.0,1.0,0.0,0.0
990,2,2025-01-02 01:40:28,2025-01-02 01:41:24,N,5.0,130,130,1.0,0.08,-8.00,0.0,0.0,0.0,0.0,NaN,-1.0,-9.00,3.0,2.0,0.0,0.0
1006,2,2025-01-02 03:39:00,2025-01-02 03:39:18,N,5.0,129,82,3.0,0.00,-0.15,0.0,0.0,0.0,0.0,NaN,-1.0,-1.15,4.0,2.0,0.0,0.0
1017,2,2025-01-02 05:12:35,2025-01-02 05:23:47,N,1.0,169,182,1.0,2.74,0.00,0.0,-0.5,0.0,0.0,NaN,-1.0,-1.50,2.0,1.0,0.0,0.0


*Negative amounts: refunds, disputes, errors? What payment types do they have? Should they count as revenue?*

In [10]:
# Biggest amounts: plausible or outliers?
df.nlargest(10, "total_amount")[["lpep_pickup_datetime","lpep_dropoff_datetime","trip_distance","total_amount","payment_type"]]

,lpep_pickup_datetime,lpep_dropoff_datetime,trip_distance,total_amount,payment_type
1122,2025-01-02 08:42:32,2025-01-02 11:37:39,86.50,371.40,2.0
25249,2025-01-18 01:50:08,2025-01-18 02:17:22,7.57,293.25,1.0
18446,2025-01-14 09:47:18,2025-01-14 10:50:51,61.89,256.94,1.0
6140,2025-01-05 19:57:08,2025-01-05 19:57:21,0.00,251.00,1.0
25411,2025-01-18 09:24:47,2025-01-18 11:49:25,122.26,242.94,1.0
2492,2025-01-03 06:01:36,2025-01-03 07:32:29,42.40,242.49,1.0
43640,2025-01-30 13:15:20,2025-01-30 13:16:34,0.01,241.39,1.0
115,2025-01-01 01:34:39,2025-01-01 02:04:21,21.47,222.00,1.0
11555,2025-01-09 13:53:40,2025-01-09 15:03:08,36.08,220.10,1.0
2662,2025-01-03 09:46:34,2025-01-03 09:46:50,0.00,217.20,1.0


## 6. Other sanity checks (optional)
Duplicates? Dropoff before pickup? Distance 0 with a high fare?

In [11]:
print("duplicates:", df.duplicated().sum())
print("dropoff < pickup:", (df["lpep_dropoff_datetime"] < df["lpep_pickup_datetime"]).sum())

duplicates: 0
dropoff < pickup: 0


## 7. Do all three months look the same?
Run the checks above for February and March too. A rule that is true for January might not hold for the others.

In [12]:
import glob
frames = {f[-15:-8]: pd.read_parquet(f) for f in sorted(glob.glob("../data/raw/*.parquet"))}
for name, d in frames.items():
    print(name, d.shape, d["lpep_pickup_datetime"].min(), d["lpep_pickup_datetime"].max(), (d["total_amount"] < 0).sum())

2025-01 (48326, 21) 2024-12-25 23:13:15 2025-02-05 18:46:24 153
2025-02 (46621, 21) 2025-01-31 22:34:50 2025-03-01 23:29:24 165
2025-03 (51539, 21) 2025-02-25 18:10:10 2025-04-01 23:41:29 160
